# 15 - Deployment and Validation Status

Read-only operational status across deployment, orchestration, validation, lineage, and optional Digital Twin evidence. Missing evidence is reported as `NOT_AVAILABLE`, never inferred as success.

In [ ]:
from pyspark.sql import functions as F

status_sources = {
    'deployment_results': ('deployment_status','observed_at'),
    'orchestration_checkpoint': ('status','observed_at'),
    'validation_results': ('status','observation_timestamp'),
    'validation_results_production': ('status','validated_at'),
    'teardown_results': ('status','observed_at'),
    'digital_twin_deployment_results': ('deployment_status','observed_at'),
}
summary_rows = []
for table_name, (status_column, timestamp_column) in status_sources.items():
    if not spark.catalog.tableExists(table_name):
        summary_rows.append((table_name,'NOT_AVAILABLE',0,None,'Evidence table does not exist; success is not inferred'))
        continue
    frame = spark.table(table_name)
    grouped = frame.groupBy(status_column).count().collect()
    latest = frame.agg(F.max(timestamp_column).alias('latest')).first()['latest']
    for row in grouped:
        summary_rows.append((table_name,str(row[status_column]),int(row['count']),latest,'Recorded evidence'))

status_schema = 'evidence_source string, status string, record_count long, latest_timestamp timestamp, detail string'
status_frame = spark.createDataFrame(summary_rows,status_schema)
display(status_frame.orderBy('evidence_source','status'))

failed_count = status_frame.filter(F.col('status').isin('FAILED','FAIL')).agg(F.sum('record_count').alias('failures')).first()['failures'] or 0
unsupported_count = status_frame.filter(F.col('status')=='SKIPPED_UNSUPPORTED').agg(F.sum('record_count').alias('unsupported')).first()['unsupported'] or 0
print('Recorded failures:',failed_count)
print('Recorded unsupported operations:',unsupported_count)
print('Status is evidence-based; NOT_AVAILABLE and SKIPPED_UNSUPPORTED are not success states.')